# Phase 1: Generate Captions for Media Dataset

**Hardware:** T4 GPU (Runtime → Change runtime type → T4 GPU)  
**What this does:** Downloads your source dataset, generates text captions
for each image using BLIP, then pushes the captioned dataset to your HF Hub.

**Before running:**
1. Set runtime to T4 GPU
2. Add secrets via the key icon (🔑) in the left sidebar — enable notebook access for each:
   - `HF_TOKEN` — HF token with write access
   - `HF_USERNAME` — your HF username (e.g. mugdhav)
3. Fill in the form fields in Cell 2:
   - Click **▶ Run** on Cell 2 — a form appears **above** the code
   - Set `SOURCE_DATASET` to your HF dataset ID (e.g. `mugdhav/media-search-demo-files`)
   - Set `IMAGE_COL` to the image column name in your dataset (default: `image`)
   - Set `VIDEO_COL` to the video column name, or leave blank if none
4. Run remaining cells top to bottom with **Shift+Enter**

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers datasets opencv-python-headless Pillow torch

In [ ]:
# Cell 2 — Config
# @title Configuration
# @markdown Fill in the dataset details below, then run this cell.
from google.colab import userdata

# Secrets (set in the key icon in left sidebar)
HF_TOKEN    = userdata.get('HF_TOKEN')
HF_USERNAME = userdata.get('HF_USERNAME')

# Form variables — edit these before running
SOURCE_DATASET = "mugdhav/media-search-demo-files" # @param {type:"string"}
IMAGE_COL      = "image" # @param {type:"string"}
VIDEO_COL      = "" # @param {type:"string"}

# Derived — no need to change
OUTPUT_DATASET = f"{HF_USERNAME}/media-search-demo-captioned"
VIDEO_COL      = VIDEO_COL if VIDEO_COL.strip() else None

# Verify
print("HF_TOKEN:   ", HF_TOKEN[:8] + "..." if HF_TOKEN else "ERROR: not set")
print("HF_USERNAME:", HF_USERNAME if HF_USERNAME else "ERROR: not set")
print("SOURCE:     ", SOURCE_DATASET)
print("OUTPUT:     ", OUTPUT_DATASET)
print("IMAGE_COL:  ", IMAGE_COL)
print("VIDEO_COL:  ", VIDEO_COL)

In [ ]:
# Cell 3 — Load and inspect dataset
from datasets import load_dataset

ds = load_dataset(SOURCE_DATASET, split="train", token=HF_TOKEN)
print("Features: ", ds.features)
print("Total items:", len(ds))
print("First item keys:", list(ds[0].keys()))
print("Image type:", type(ds[0][IMAGE_COL]))

In [ ]:
# Cell 4 — Load BLIP captioning model
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

cap_proc  = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-large", token=HF_TOKEN
)
cap_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-large",
    torch_dtype=torch.float16,
    token=HF_TOKEN
).to(device).eval()
print("BLIP loaded")

In [ ]:
# Cell 5 — Caption and frame extraction helpers
import io, cv2, numpy as np
from PIL import Image

def caption_image(pil_img):
    inputs = cap_proc(images=pil_img, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = cap_model.generate(**inputs, max_new_tokens=128)
    return cap_proc.decode(ids[0], skip_special_tokens=True)

def frames_from_video_bytes(video_bytes, n=5):
    tmp = "/tmp/vid.mp4"
    with open(tmp, "wb") as f:
        f.write(video_bytes)
    cap   = cv2.VideoCapture(tmp)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []
    frames = []
    for idx in np.linspace(0, total - 1, min(n, total), dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames

print("Helpers defined")

In [ ]:
# Cell 6 — Generate captions and push to Hub
from datasets import Dataset, Image as HFImage

records = []

for i, item in enumerate(ds):
    try:
        if IMAGE_COL and IMAGE_COL in item and item[IMAGE_COL] is not None:
            img = item[IMAGE_COL]
            if isinstance(img, Image.Image):
                img = img.convert("RGB")
            else:
                img = Image.open(io.BytesIO(img)).convert("RGB")
            caption = caption_image(img)
            records.append({"image": img, "caption": caption, "source_type": "image"})
            print(f"[{i+1}/{len(ds)}] {caption[:80]}")

        elif VIDEO_COL and VIDEO_COL in item and item[VIDEO_COL] is not None:
            frames = frames_from_video_bytes(item[VIDEO_COL])
            if frames:
                mid     = frames[len(frames) // 2]
                caption = caption_image(mid)
                records.append({"image": mid, "caption": caption, "source_type": "video"})
                print(f"[{i+1}/{len(ds)}] (video) {caption[:80]}")
        else:
            print(f"[SKIP] item {i}: no image or video found")

    except Exception as e:
        print(f"[WARN] item {i}: {e}")

print(f"\nRecords generated: {len(records)}")

if len(records) > 0:
    out = Dataset.from_list(records).cast_column("image", HFImage())
    out.push_to_hub(OUTPUT_DATASET, token=HF_TOKEN)
    print(f"Done — pushed {len(out)} records to {OUTPUT_DATASET}")
else:
    print("No records generated — check warnings above.")

In [ ]:
# Cell 7 — Verify pushed dataset
from datasets import load_dataset

check = load_dataset(OUTPUT_DATASET, split="train[:3]", token=HF_TOKEN)
print(f"Dataset on Hub: {len(check)} sample rows")
for i, row in enumerate(check):
    print(f"  [{i+1}] {row['caption']}")